In [47]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

In [2]:
# Set plot style
sns.set_theme(style="whitegrid")

# --- Cell 1: Load Data & Executive KPIs ---

In [4]:
# 1. Load the 3-year historical dataset
df = pd.read_csv("dialysis_billing_data.csv")

In [5]:
# Ensure Trans_Date is datetime
df["Trans_Date"] = pd.to_datetime(df["Trans_Date"])

In [6]:
# 2. Calculate Gross Profit per line item
df["Profit"] = df["Amount_Excl"] - df["Cost"]

In [7]:
# 3. High-Level Financial KPIs
total_gross_revenue = df["Amount_Excl"].sum()
total_cost = df["Cost"].sum()
total_net_profit = df["Profit"].sum()
overall_margin = (
    (total_net_profit / total_gross_revenue) * 100
    if total_gross_revenue
    else 0
)

In [8]:
# 4. Operational KPIs
# Unique sessions = unique non-reversed procedure invoices
total_sessions = df[df["Category"].isin(["Chronic", "Acute"])][
    "Invoice_No"
].nunique()
total_patients = df["Patient_ID"].nunique()
reversal_count = df[df["Is_Reversal"] == True]["Invoice_No"].nunique()

# Display Summary
print("==========================================")
print("       EXECUTIVE SUMMARY (3-YEAR)         ")
print("==========================================")
print(f"Total Billed Revenue : ${total_gross_revenue:,.2f}")
print(f"Total Direct Cost    : ${total_cost:,.2f}")
print(f"Total Net Profit     : ${total_net_profit:,.2f}")
print(f"Overall Gross Margin : {overall_margin:.1f}%")
print("------------------------------------------")
print(f"Total Dialysis Sessions : {total_sessions:,}")
print(f"Unique Patients Treated  : {total_patients}")
print(f"Reversed Invoices Count  : {reversal_count}")
print("==========================================")

       EXECUTIVE SUMMARY (3-YEAR)         
Total Billed Revenue : $19,490,106.73
Total Direct Cost    : $1,407,015.13
Total Net Profit     : $18,083,091.60
Overall Gross Margin : 92.8%
------------------------------------------
Total Dialysis Sessions : 6,763
Unique Patients Treated  : 150
Reversed Invoices Count  : 263


# --- CELL 2: OPERATING EXPENSES & REALISTIC NET PROFIT ---

In [10]:
# 1. Define estimated fixed overhead percentage (e.g., 55% of revenue for staff/facility/machines)
OPEX_PERCENTAGE = 0.55

In [11]:
# 2. Calculate Operating Overhead (OpEx)
df_valid = df[
    df["Is_Reversal"] == False
]  # Or compute on net revenue across all rows
net_billed_revenue = df["Amount_Excl"].sum()
direct_stock_cost = df["Cost"].sum()

gross_profit = net_billed_revenue - direct_stock_cost
estimated_opex = net_billed_revenue * OPEX_PERCENTAGE
true_net_profit = gross_profit - estimated_opex

gross_margin_pct = (gross_profit / net_billed_revenue) * 100
net_margin_pct = (true_net_profit / net_billed_revenue) * 100

print("==========================================")
print("     FINANCIAL MARGIN RECONCILIATION      ")
print("==========================================")
print(f"Net Billed Revenue   : ${net_billed_revenue:,.2f}")
print(f"Direct Stock Cost    : ${direct_stock_cost:,.2f}")
print(f"------------------------------------------")
print(
    f"Gross Profit         : ${gross_profit:,.2f} ({gross_margin_pct:.1f}% Margin)"
)
print(f"Est. Facility OpEx   : ${estimated_opex:,.2f} (55% of Revenue)")
print(f"------------------------------------------")
print(
    f"True Net Profit      : ${true_net_profit:,.2f} ({net_margin_pct:.1f}% Net Margin)"
)
print("==========================================")

     FINANCIAL MARGIN RECONCILIATION      
Net Billed Revenue   : $19,490,106.73
Direct Stock Cost    : $1,407,015.13
------------------------------------------
Gross Profit         : $18,083,091.60 (92.8% Margin)
Est. Facility OpEx   : $10,719,558.70 (55% of Revenue)
------------------------------------------
True Net Profit      : $7,363,532.90 (37.8% Net Margin)


# --- CELL 3: MONTHLY TIME-SERIES AGGREGATIONS ---

In [27]:
# 1. Load data & prepare date fields
df = pd.read_csv("dialysis_historical_3yr.csv")
df["Trans_Date"] = pd.to_datetime(df["Trans_Date"])
df["Year"] = df["Trans_Date"].dt.year
df["Month_Name"] = df["Trans_Date"].dt.strftime("%b")
df["Month_Num"] = df["Trans_Date"].dt.month

# List of available months for proper sorting
months_order = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]


@interact(Select_Year=[2023, 2024, 2025, 2026])
def plot_interactive_regional_revenue(Select_Year):
    # Filter dataset for selected year
    df_filtered = df[df["Year"] == Select_Year].copy()

    # Aggregate monthly revenue by Month and Service Centre
    monthly_centre = (
        df_filtered.groupby(["Month_Num", "Month_Name", "Service_Centre"])[
            "Amount_Excl"
        ]
        .sum()
        .reset_index()
        .sort_values("Month_Num")
    )

    # Create Line Chart styled like Seaborn/Matplotlib
    fig = px.line(
        monthly_centre,
        x="Month_Name",
        y="Amount_Excl",
        color="Service_Centre",
        markers=True,
        category_orders={"Month_Name": months_order},
        # Seaborn 'deep' palette colors
        color_discrete_sequence=["#4C72B0", "#DD8452", "#55A868"],
        labels={
            "Amount_Excl": "Billed Revenue",
            "Month_Name": "Month",
            "Service_Centre": "Region",
        },
    )

    # Customize Hover Tooltip (Mouseover option)
    fig.update_traces(
        hovertemplate="<b>Region:</b> %{fullData.name}<br>"
        + "<b>Date:</b> %{x} "
        + str(Select_Year)
        + "<br>"
        + "<b>Revenue:</b> $%{y:,.2f}<extra></extra>"
    )

    # Seaborn-style clean whitegrid layout
    fig.update_layout(
        title=dict(
            text=f"Monthly Revenue by Region — {Select_Year}",
            font=dict(size=16, family="Arial"),
        ),
        xaxis_title="Month",
        yaxis_title="Billed Revenue ($)",
        yaxis_tickprefix="$",
        yaxis_tickformat=",",
        plot_bgcolor="white",
        hovermode="x unified",  # Compares all regions together on hover
        legend=dict(
            title="Service Centre",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
    )

    # Add light gridlines like Seaborn
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#E5E5E5")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#E5E5E5")

    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=(2023, 2024, 2025, 2026), value=2023), Outpu…

In [31]:
@interact(Select_Year=[2023, 2024, 2025, 2026])
def plot_session_counts(Select_Year):
    # Filter dataset for selected year & non-stock procedure rows
    df_filtered = df[
        (df["Year"] == Select_Year) & (df["Category"].isin(["Chronic", "Acute"]))
    ].copy()

    # 1. Calculate monthly counts per category (Chronic & Acute)
    cat_counts = (
        df_filtered.groupby(["Month_Num", "Month_Name", "Category"])[
            "Invoice_No"
        ]
        .nunique()
        .reset_index()
    )
    cat_counts.rename(columns={"Invoice_No": "Session_Count"}, inplace=True)

    # 2. Calculate monthly total counts (Chronic + Acute combined)
    total_counts = (
        df_filtered.groupby(["Month_Num", "Month_Name"])["Invoice_No"]
        .nunique()
        .reset_index()
    )
    total_counts.rename(columns={"Invoice_No": "Session_Count"}, inplace=True)
    total_counts["Category"] = "Total (Chronic + Acute)"

    # 3. Combine into a single table for plotting
    monthly_sessions = (
        pd.concat([cat_counts, total_counts], ignore_index=True)
        .sort_values("Month_Num")
    )

    # 4. Create Line Chart styled like Seaborn
    fig = px.line(
        monthly_sessions,
        x="Month_Name",
        y="Session_Count",
        color="Category",
        markers=True,
        category_orders={"Month_Name": months_order},
        # Distinct Seaborn color palette: Blue (Chronic), Orange (Acute), Slate Gray (Total)
        color_discrete_map={
            "Chronic": "#4C72B0",
            "Acute": "#DD8452",
            "Total (Chronic + Acute)": "#555555",
        },
        labels={
            "Session_Count": "Sessions Count",
            "Month_Name": "Month",
            "Category": "Session Type",
        },
    )

    # Make the 'Total' line slightly thicker and dashed for visual clarity
    fig.for_each_trace(
        lambda trace: trace.update(line=dict(dash="dash", width=3))
        if trace.name == "Total (Chronic + Acute)"
        else trace.update(line=dict(width=2))
    )

    # Custom Hover Tooltip
    fig.update_traces(
        hovertemplate="<b>Session Type:</b> %{fullData.name}<br>"
        + "<b>Date:</b> %{x} "
        + str(Select_Year)
        + "<br>"
        + "<b>Sessions Count:</b> %{y:,}<extra></extra>"
    )

    # Seaborn whitegrid layout
    fig.update_layout(
        title=dict(
            text=f"Monthly Session Counts (Chronic vs Acute vs Total) — {Select_Year}",
            font=dict(size=16, family="Arial"),
        ),
        xaxis_title="Month",
        yaxis_title="Number of Sessions",
        yaxis_tickformat=",",
        plot_bgcolor="white",
        hovermode="x unified",
        legend=dict(
            title="Session Type",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
    )

    # Gridlines
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#E5E5E5")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#E5E5E5")

    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=(2023, 2024, 2025, 2026), value=2023), Outpu…

In [33]:
@interact(Select_Year=[2023, 2024, 2025, 2026])
def plot_medical_aid_revenue(Select_Year):
    # Filter dataset for selected year
    df_filtered = df[df["Year"] == Select_Year].copy()

    # Aggregate monthly revenue by Month and Medical Aid
    med_aid_monthly = (
        df_filtered.groupby(["Month_Num", "Month_Name", "Medical_Aid"])[
            "Amount_Excl"
        ]
        .sum()
        .reset_index()
        .sort_values("Month_Num")
    )

    # Create Grouped Bar Chart
    fig = px.bar(
        med_aid_monthly,
        x="Month_Name",
        y="Amount_Excl",
        color="Medical_Aid",
        barmode="group",  # Groups bars side-by-side for easy comparison
        category_orders={"Month_Name": months_order},
        # Custom distinct palette matching Seaborn theme
        color_discrete_sequence=["#4C72B0", "#DD8452", "#55A868"],
        labels={
            "Amount_Excl": "Billed Revenue ($)",
            "Month_Name": "Month",
            "Medical_Aid": "Medical Aid / Payor",
        },
    )

    # Custom Hover Tooltip
    fig.update_traces(
        hovertemplate="<b>Payor:</b> %{fullData.name}<br>"
        + "<b>Date:</b> %{x} "
        + str(Select_Year)
        + "<br>"
        + "<b>Revenue:</b> $%{y:,.2f}<extra></extra>"
    )

    # Clean whitegrid styling
    fig.update_layout(
        title=dict(
            text=f"Monthly Billed Revenue by Medical Aid — {Select_Year}",
            font=dict(size=16, family="Arial"),
        ),
        xaxis_title="Month",
        yaxis_title="Billed Revenue ($)",
        yaxis_tickprefix="$",
        yaxis_tickformat=",",
        plot_bgcolor="white",
        legend=dict(
            title="Medical Aid",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
        ),
    )

    # Add light vertical/horizontal gridlines
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#E5E5E5")

    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=(2023, 2024, 2025, 2026), value=2023), Outpu…

In [39]:
@interact(
    Select_Year=["All Years", 2023, 2024, 2025, 2026],
    Scale_Type=["Linear (Default)", "Logarithmic (Equal Visibility)"],
)
def plot_stock_profitability_v2(Select_Year, Scale_Type):
    if Select_Year == "All Years":
        stock_df = df[df["Category"] == "Stock"].copy()
    else:
        stock_df = df[
            (df["Category"] == "Stock") & (df["Year"] == Select_Year)
        ].copy()

    stock_summary = (
        stock_df.groupby("Description")
        .agg(
            Total_Qty=("Quantity", "sum"),
            Total_Revenue=("Amount_Excl", "sum"),
            Total_Cost=("Cost", "sum"),
        )
        .reset_index()
    )

    stock_summary["Gross_Profit"] = (
        stock_summary["Total_Revenue"] - stock_summary["Total_Cost"]
    )
    stock_summary["Margin_Pct"] = (
        stock_summary["Gross_Profit"] / stock_summary["Total_Revenue"]
    ) * 100
    stock_summary = stock_summary.sort_values(by="Gross_Profit", ascending=True)

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            y=stock_summary["Description"],
            x=stock_summary["Total_Cost"],
            name="Direct Stock Cost",
            orientation="h",
            marker_color="#DD8452",
            customdata=stock_summary[["Total_Qty", "Margin_Pct"]],
            hovertemplate="<b>Item:</b> %{y}<br><b>Cost:</b> $%{x:,.2f}<br><b>Qty:</b> %{customdata[0]:,}<extra></extra>",
        )
    )

    fig.add_trace(
        go.Bar(
            y=stock_summary["Description"],
            x=stock_summary["Gross_Profit"],
            name="Gross Profit",
            orientation="h",
            marker_color="#55A868",
            customdata=stock_summary[["Total_Qty", "Margin_Pct"]],
            hovertemplate="<b>Item:</b> %{y}<br><b>Gross Profit:</b> $%{x:,.2f}<br><b>Margin:</b> %{customdata[1]:.1f}%<br><b>Qty:</b> %{customdata[0]:,}<extra></extra>",
        )
    )

    is_log = Scale_Type == "Logarithmic (Equal Visibility)"

    fig.update_layout(
        barmode="stack",
        title=dict(
            text=f"Stock Consumables Profitability ({Scale_Type}) — {Select_Year}",
            font=dict(size=15),
        ),
        xaxis_title="Dollar Amount ($)" if not is_log else "Dollar Amount (Log Scale)",
        xaxis_type="log" if is_log else "linear",
        xaxis_tickprefix="$",
        xaxis_tickformat=",",
        plot_bgcolor="white",
        height=450,
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
    )

    fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=('All Years', 2023, 2024, 2025, 2026), value…

In [41]:
@interact(Select_Year=["All Years", 2023, 2024, 2025, 2026])
def plot_stock_profit_split(Select_Year):
    if Select_Year == "All Years":
        stock_df = df[df["Category"] == "Stock"].copy()
    else:
        stock_df = df[
            (df["Category"] == "Stock") & (df["Year"] == Select_Year)
        ].copy()

    # Classify items as High-Value Equipment (Dialyzer) vs General Consumables
    stock_df["Item_Group"] = stock_df["Description"].apply(
        lambda x: "Dialyzers (High-Value)"
        if "DIALYZER" in str(x).upper()
        else "General Sundries & Meds"
    )

    stock_df["Gross_Profit"] = stock_df["Amount_Excl"] - stock_df["Cost"]

    group_summary = (
        stock_df.groupby("Item_Group")["Gross_Profit"].sum().reset_index()
    )

    fig = px.pie(
        group_summary,
        values="Gross_Profit",
        names="Item_Group",
        title=f"Stock Profit Share: High-Value Equipment vs General Supplies ({Select_Year})",
        hole=0.5,  # Creates Donut Chart
        color_discrete_sequence=["#55A868", "#4C72B0"],
    )

    fig.update_traces(
        textinfo="percent+label",
        hovertemplate="<b>%{label}</b><br>Gross Profit: $%{value:,.2f}<extra></extra>",
    )

    fig.update_layout(
        plot_bgcolor="white",
        showlegend=False,
        height=380,
    )

    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=('All Years', 2023, 2024, 2025, 2026), value…

In [55]:
st.markdown(
    "**Key Finding:** 77% of consumable stock margin is concentrated in Dialyzer units alone, making supplier contract negotiations for Dialyzers the primary lever for inventory cost reduction."
)

DeltaGenerator()

In [43]:
@interact(Select_Year=[2023, 2024, 2025, 2026])
def plot_stock_cost_per_session(Select_Year):
    df_filtered = df[df["Year"] == Select_Year].copy()

    # 1. Total stock cost per month and region
    stock_monthly = (
        df_filtered[df_filtered["Category"] == "Stock"]
        .groupby(["Month_Num", "Month_Name", "Service_Centre"])["Cost"]
        .sum()
        .reset_index()
    )

    # 2. Total treatment session counts per month and region
    session_monthly = (
        df_filtered[df_filtered["Category"].isin(["Chronic", "Acute"])]
        .groupby(["Month_Num", "Month_Name", "Service_Centre"])["Invoice_No"]
        .nunique()
        .reset_index()
    )

    # 3. Merge & calculate Stock Cost per Session
    merged = pd.merge(
        stock_monthly,
        session_monthly,
        on=["Month_Num", "Month_Name", "Service_Centre"],
    )
    merged["Cost_Per_Session"] = merged["Cost"] / merged["Invoice_No"]
    merged = merged.sort_values("Month_Num")

    fig = px.line(
        merged,
        x="Month_Name",
        y="Cost_Per_Session",
        color="Service_Centre",
        markers=True,
        category_orders={"Month_Name": months_order},
        color_discrete_sequence=["#4C72B0", "#DD8452", "#55A868"],
        labels={
            "Cost_Per_Session": "Stock Cost / Session ($)",
            "Month_Name": "Month",
            "Service_Centre": "Clinic Location",
        },
    )

    fig.update_traces(
        hovertemplate="<b>Clinic:</b> %{fullData.name}<br>"
        + "<b>Date:</b> %{x} "
        + str(Select_Year)
        + "<br>"
        + "<b>Avg Stock Cost/Session:</b> $%{y:,.2f}<extra></extra>"
    )

    fig.update_layout(
        title=dict(
            text=f"Average Consumable Stock Cost per Treatment Session — {Select_Year}",
            font=dict(size=15),
        ),
        xaxis_title="Month",
        yaxis_title="Stock Cost per Session ($)",
        yaxis_tickprefix="$",
        yaxis_tickformat=",",
        plot_bgcolor="white",
        hovermode="x unified",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
    )

    fig.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
    fig.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

    fig.show()

interactive(children=(Dropdown(description='Select_Year', options=(2023, 2024, 2025, 2026), value=2023), Outpu…

In [53]:
st.markdown(
    "**Key Finding:** Consumable stock cost per treatment session remains highly stable at ~$195–$200/session across all three centers, showing consistent clinical inventory management without noticeable waste."
)

DeltaGenerator()

# --- CELL 4: REVERSAL & BILLING LEAKAGE ANALYSIS ---

In [60]:
@interact(Select_Year=[2023, 2024, 2025, 2026])
def plot_reversal_analysis(Select_Year):
    df_filtered = df[df["Year"] == Select_Year].copy()

    # 1. Aggregate monthly billed vs reversed amounts
    monthly_leakage = (
        df_filtered.groupby(["Month_Num", "Month_Name", "Service_Centre"])
        .agg(
            Total_Billed=("Amount_Excl", lambda x: x[x > 0].sum()),
            Reversed_Amount=("Amount_Excl", lambda x: abs(x[x < 0].sum())),
            Reversal_Count=("Is_Reversal", lambda x: (x == True).sum()),
        )
        .reset_index()
    )

    monthly_leakage["Leakage_Pct"] = (
        monthly_leakage["Reversed_Amount"] / monthly_leakage["Total_Billed"]
    ) * 100
    monthly_leakage = monthly_leakage.sort_values("Month_Num")

    # Chart 1: Monthly Financial Leakage Rate (%)
    fig1 = px.line(
        monthly_leakage,
        x="Month_Name",
        y="Leakage_Pct",
        color="Service_Centre",
        markers=True,
        category_orders={"Month_Name": months_order},
        color_discrete_sequence=["#4C72B0", "#DD8452", "#55A868"],
        labels={
            "Leakage_Pct": "Financial Leakage Rate (%)",
            "Month_Name": "Month",
            "Service_Centre": "Clinic Location",
        },
    )

    fig1.update_traces(
        hovertemplate="<b>Clinic:</b> %{fullData.name}<br>"
        + "<b>Date:</b> %{x} "
        + str(Select_Year)
        + "<br>"
        + "<b>Leakage Rate:</b> %{y:.2f}%<extra></extra>"
    )

    fig1.update_layout(
        title=dict(
            text=f"Monthly Revenue Leakage Rate (% Billed Revenue Reversed) — {Select_Year}",
            font=dict(size=15),
        ),
        xaxis_title="Month",
        yaxis_title="Revenue Leakage (%)",
        yaxis_ticksuffix="%",
        plot_bgcolor="white",
        hovermode="x unified",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
    )
    fig1.update_xaxes(showgrid=True, gridcolor="#E5E5E5")
    fig1.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

    fig1.show()

    # Chart 2: Reversal Value by Medical Aid and Category
    reversal_df = df_filtered[df_filtered["Is_Reversal"] == True].copy()
    reversal_df["Abs_Amount"] = reversal_df["Amount_Excl"].abs()

    med_reversals = (
        reversal_df.groupby(["Medical_Aid", "Category"])["Abs_Amount"]
        .sum()
        .reset_index()
    )

    fig2 = px.bar(
        med_reversals,
        x="Medical_Aid",
        y="Abs_Amount",
        color="Category",
        barmode="stack",
        color_discrete_sequence=["#DD8452", "#4C72B0", "#55A868"],
        labels={
            "Abs_Amount": "Total Reversed Value ($)",
            "Medical_Aid": "Medical Aid / Payor",
            "Category": "Billing Category",
        },
    )

    fig2.update_traces(
        hovertemplate="<b>Payor:</b> %{x}<br>"
        + "<b>Category:</b> %{fullData.name}<br>"
        + "<b>Reversed Value:</b> $%{y:,.2f}<extra></extra>"
    )

    fig2.update_layout(
        title=dict(
            text=f"Total Value of Reversals by Payor & Billing Category — {Select_Year}",
            font=dict(size=15),
        ),
        xaxis_title="Medical Aid",
        yaxis_title="Reversed Value ($)",
        yaxis_tickprefix="$",
        yaxis_tickformat=",",
        plot_bgcolor="white",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
    )
    fig2.update_xaxes(showgrid=False)
    fig2.update_yaxes(showgrid=True, gridcolor="#E5E5E5")

    fig2.show()

interactive(children=(Dropdown(description='Select_Year', options=(2023, 2024, 2025, 2026), value=2023), Outpu…

In [62]:
st.markdown(
    "**Key Finding:** Reversals consume between 2.9% and 6.4% of total monthly billed revenue across clinics, representing avoidable financial leakage."
)

DeltaGenerator()

In [64]:
st.markdown(
    "**Key Finding:** The bulk of monetary reversals stem from Chronic Dialysis sessions under PRIVATE STATE payors, suggesting potential delays or errors in pre-authorization confirmation prior to procedure entry."
)

DeltaGenerator()

In [66]:
st.markdown(
    "**Key Finding:** Implementing automated claim validation before submitting state medical aid invoices can reduce monthly administrative rework by up to ~6%."
)

DeltaGenerator()